# 05b — TabDDPM Ablation: QuantileTransformer por columna + clipping

**Motivación:** El notebook 05 produce muestras con distribuciones marginales
completamente distorsionadas (JSD mediana = 0.644), pese a que las correlaciones
entre variables se preservan bien (Δρ = 0.056).

La causa raíz identificada: el difusor genera valores en el espacio normalizado
fuera del rango de entrenamiento, y el `QuantileTransformer` de sklearn extrapola
linealmente desde los cuantiles extremos, produciendo valores absurdos en espacio
original (e.g. heart_rate_mean = -1398 en lugar de ~87).

**Cambios respecto a nb05:**
1. `PerColumnTabularPreprocessor`: un `QuantileTransformer` independiente por columna numérica.
2. Almacenamiento del rango [min, max] de cada columna en espacio normalizado.
3. **Clipping** de las muestras generadas a ese rango antes del inverse_transform.

El modelo, hiperparámetros y loop de entrenamiento son idénticos a nb05.
La salida se guarda en `tabddpm_v2_samples.parquet` para no sobreescribir los resultados originales.

## 0. Imports y configuración

In [ ]:
import sys, warnings, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from tqdm import tqdm

import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import QuantileTransformer, OrdinalEncoder
from scipy.stats import wasserstein_distance

ROOT      = Path("..")
PROCESSED = ROOT / "data" / "processed"
SYNTHETIC = ROOT / "data" / "synthetic"
MODELS    = ROOT / "models"
REPORTS   = ROOT / "reports"
for d in [SYNTHETIC, MODELS, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "src"))
from models.tabddpm import TabDDPMDenoiser, CosineScheduler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42)
np.random.seed(42)
sns.set_theme(style="whitegrid", palette="muted")

## 1. Carga y clasificación de columnas

Idéntico a nb05.

In [ ]:
tab = pd.read_parquet(PROCESSED / "tabular_48h.parquet")
print(f"Shape: {tab.shape}")

TARGET   = "hospital_expire_flag"
id_cols  = [c for c in tab.columns if c.endswith("_id")]

feature_cols = [c for c in tab.columns if c not in id_cols + [TARGET]]

binary_cols = [
    c for c in feature_cols
    if tab[c].dropna().nunique() <= 2 and tab[c].dtype != object
]
cat_cols = [
    c for c in feature_cols
    if c not in binary_cols and tab[c].dtype == object
]
num_cols = [
    c for c in feature_cols
    if c not in binary_cols + cat_cols
]

print(f"\nFeatures totales:      {len(feature_cols)}")
print(f"  Numéricas:           {len(num_cols)}")
print(f"  Binarias:            {len(binary_cols)}")
print(f"  Categóricas texto:   {len(cat_cols)}  → {cat_cols}")
print(f"\nTarget: {TARGET}  ({tab[TARGET].mean()*100:.1f}% positivos)")

## 2. Preprocesamiento por columna (cambio principal)

En nb05, un único `QuantileTransformer` se ajusta sobre las 115 columnas numéricas
simultáneamente. Aunque sklearn lo trata de forma independiente por columna, el
problema crítico es que no se almacena el rango de cada columna en espacio
normalizado para poder hacer clipping en la generación.

La clase `PerColumnTabularPreprocessor` introduce:
- Un `QuantileTransformer` independiente por columna numérica.
- Almacenamiento del `[min, max]` en espacio normalizado tras el fit.
- Clipping opcional antes de `inverse_transform`.

In [ ]:
class PerColumnTabularPreprocessor:
    """
    Preprocesador para difusión gaussiana con un QuantileTransformer por columna
    numérica y clipping en espacio normalizado antes de la transformada inversa.

    El clipping es el cambio crítico respecto a nb05: evita que el QuantileTransformer
    extrapole linealmente para valores fuera del rango de entrenamiento, lo que
    producía valores absurdos (e.g. heart_rate_mean = -1398).
    """

    def __init__(self, num_cols: list, binary_cols: list, cat_cols: list):
        self.num_cols    = num_cols
        self.binary_cols = binary_cols
        self.cat_cols    = cat_cols
        self.input_dim   = len(num_cols) + len(binary_cols) + len(cat_cols)
        self.transformers: dict = {}      # col → QuantileTransformer
        self.clip_min: np.ndarray = None  # rango mínimo por columna en espacio normalizado
        self.clip_max: np.ndarray = None
        self.cat_encoder = (
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
            if cat_cols else None
        )

    def fit(self, df: pd.DataFrame) -> "PerColumnTabularPreprocessor":
        for col in self.num_cols:
            qt = QuantileTransformer(
                n_quantiles=min(1000, len(df)),
                output_distribution="normal",
                random_state=42,
            )
            qt.fit(df[[col]])
            self.transformers[col] = qt
        if self.cat_encoder:
            self.cat_encoder.fit(df[self.cat_cols])
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        parts = []
        for col in self.num_cols:
            parts.append(
                self.transformers[col].transform(df[[col]]).astype(np.float32)
            )
        if self.binary_cols:
            parts.append(df[self.binary_cols].values.astype(np.float32) * 2 - 1)
        if self.cat_encoder:
            enc = self.cat_encoder.transform(df[self.cat_cols]).astype(np.float32)
            for i in range(enc.shape[1]):
                n = len(self.cat_encoder.categories_[i])
                enc[:, i] = enc[:, i] / max(n - 1, 1) * 2 - 1
            parts.append(enc)
        return np.concatenate(parts, axis=1)

    def fit_transform(self, df: pd.DataFrame) -> np.ndarray:
        X = self.fit(df).transform(df)
        # Guardar rango de entrenamiento en espacio normalizado para clipping
        self.clip_min = X.min(axis=0)
        self.clip_max = X.max(axis=0)
        return X

    def inverse_transform(self, arr: np.ndarray, clip: bool = True) -> pd.DataFrame:
        if clip and self.clip_min is not None:
            arr = np.clip(arr, self.clip_min, self.clip_max)

        result = {}
        idx = 0

        for col in self.num_cols:
            val = self.transformers[col].inverse_transform(arr[:, idx:idx + 1])
            result[col] = val[:, 0]
            idx += 1

        if self.binary_cols:
            for col in self.binary_cols:
                result[col] = np.clip(np.round((arr[:, idx] + 1) / 2), 0, 1).astype(int)
                idx += 1

        if self.cat_encoder:
            n   = len(self.cat_cols)
            enc = arr[:, idx:idx + n].copy()
            for i in range(n):
                n_cats    = len(self.cat_encoder.categories_[i])
                enc[:, i] = np.clip(
                    np.round((enc[:, i] + 1) / 2 * (n_cats - 1)), 0, n_cats - 1
                )
            inv = self.cat_encoder.inverse_transform(enc.astype(int))
            for i, col in enumerate(self.cat_cols):
                result[col] = inv[:, i]

        return pd.DataFrame(result)


preprocessor = PerColumnTabularPreprocessor(num_cols, binary_cols, cat_cols)
X = preprocessor.fit_transform(tab)
y = tab[TARGET].values.astype(np.int64)

print(f"Tensor de entrada: {X.shape}")
print(f"Rango normalizado: [{X.min():.2f}, {X.max():.2f}]")
print(f"Rango de clipping guardado: clip_min[0]={preprocessor.clip_min[0]:.3f}, clip_max[0]={preprocessor.clip_max[0]:.3f}")
print(f"Labels: {np.bincount(y)}  (0=superviviente, 1=fallecido)")

# Serializar preprocesador para evaluación posterior
joblib.dump(preprocessor, MODELS / "tabddpm_v2_preprocessor.pkl")
print("Preprocesador guardado: models/tabddpm_v2_preprocessor.pkl")

## 3. Modelo y scheduler

Idéntico a nb05: misma arquitectura, mismos hiperparámetros.

In [ ]:
T        = 1000
N_EPOCHS = 1000
BATCH    = 4096
LR       = 3e-4

INPUT_DIM = preprocessor.input_dim
print(f"input_dim: {INPUT_DIM}")

model = TabDDPMDenoiser(
    input_dim=INPUT_DIM,
    hidden_dims=(512, 512, 512, 512),
    time_emb_dim=128,
    num_classes=2,
    dropout=0.0,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {n_params:,}")

diffusion = CosineScheduler(T=T, s=0.008).to(DEVICE)

X_tensor = torch.from_numpy(X).float()
y_tensor = torch.from_numpy(y).long()
dataset  = TensorDataset(X_tensor, y_tensor)
loader   = DataLoader(dataset, batch_size=BATCH, shuffle=True, drop_last=True, num_workers=0)

## 4. Entrenamiento

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
lr_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

losses = []
t0 = time.time()

for epoch in tqdm(range(1, N_EPOCHS + 1), desc="Entrenando TabDDPM v2"):
    model.train()
    epoch_loss = 0.0
    for x_batch, y_batch in loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        optimizer.zero_grad()
        loss = diffusion.training_loss(model, x_batch, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(loader)
    losses.append(avg_loss)
    lr_sched.step()

    if epoch % 100 == 0:
        elapsed = (time.time() - t0) / 60
        tqdm.write(f"Epoch {epoch:>4}/{N_EPOCHS}  loss: {avg_loss:.5f}  ({elapsed:.1f} min)")

total_time = (time.time() - t0) / 60
print(f"\nEntrenamiento completado en {total_time:.1f} min.")

torch.save(
    {"model_state": model.state_dict(), "losses": losses,
     "input_dim": INPUT_DIM, "T": T},
    MODELS / "tabddpm_v2_checkpoint.pt"
)
print("Checkpoint guardado: models/tabddpm_v2_checkpoint.pt")

## 5. Generación de muestras sintéticas

In [ ]:
n_fallecidos     = int(tab[TARGET].sum())
n_supervivientes = len(tab) - n_fallecidos
N_SAMPLES        = len(tab)

y_gen = torch.cat([
    torch.zeros(n_supervivientes, dtype=torch.long),
    torch.ones(n_fallecidos, dtype=torch.long),
]).to(DEVICE)

print(f"Generando {N_SAMPLES:,} muestras (T={T} pasos de denoising)...")
t0 = time.time()

x_gen    = diffusion.sample(model, N_SAMPLES, INPUT_DIM, DEVICE, y=y_gen)
x_gen_np = x_gen.cpu().numpy()

print(f"Inferencia completada en {(time.time()-t0)/60:.1f} min.")
print(f"Rango generado (pre-clip): [{x_gen_np.min():.2f}, {x_gen_np.max():.2f}]")

# inverse_transform con clipping activado
synth_df = preprocessor.inverse_transform(x_gen_np, clip=True)
synth_df[TARGET] = y_gen.cpu().numpy()

synth_df.to_parquet(SYNTHETIC / "tabddpm_v2_samples.parquet", index=False)
print(f"Guardado: data/synthetic/tabddpm_v2_samples.parquet  {synth_df.shape}")
print(f"Mortalidad generada: {synth_df[TARGET].mean()*100:.1f}%  (real: {tab[TARGET].mean()*100:.1f}%)")

## 6. Comparación rápida: v1 vs v2

Estadísticos descriptivos y JSD por columna para las variables más problemáticas en nb05.

In [ ]:
tab_v1 = pd.read_parquet(SYNTHETIC / "tabddpm_samples.parquet")
tab_v2 = synth_df

key_cols = [
    "heart_rate_mean", "sbp_mean", "lactate_mean", "creatinine_mean",
    "albumin_mean", "troponin_mean", "gcs_eye_min", "age", "los"
]
key_cols = [c for c in key_cols if c in tab.columns]

rows = []
for col in key_cols:
    rows.append({
        "variable":   col,
        "real_mean":  round(tab[col].mean(), 3),
        "real_std":   round(tab[col].std(), 3),
        "v1_mean":    round(tab_v1[col].mean(), 3) if col in tab_v1.columns else None,
        "v1_std":     round(tab_v1[col].std(), 3)  if col in tab_v1.columns else None,
        "v2_mean":    round(tab_v2[col].mean(), 3) if col in tab_v2.columns else None,
        "v2_std":     round(tab_v2[col].std(), 3)  if col in tab_v2.columns else None,
    })

cmp = pd.DataFrame(rows).set_index("variable")
print(cmp.to_string())

In [ ]:
# JSD por columna: real vs v1 vs v2
def jsd_col(real: pd.Series, synth: pd.Series, n_bins: int = 50) -> float:
    lo, hi = real.min(), real.max()
    if lo == hi:
        return 0.0
    bins   = np.linspace(lo, hi, n_bins + 1)
    p, _   = np.histogram(real.dropna(),  bins=bins, density=True)
    q, _   = np.histogram(synth.dropna(), bins=bins, density=True)
    p      = p + 1e-10
    q      = q + 1e-10
    p     /= p.sum()
    q     /= q.sum()
    m      = 0.5 * (p + q)
    kl     = lambda a, b: np.sum(a * np.log(a / b))
    return float(0.5 * kl(p, m) + 0.5 * kl(q, m))

num_eval = [c for c in num_cols if c in tab_v2.columns]
jsd_v1   = [jsd_col(tab[c], tab_v1[c]) for c in num_eval if c in tab_v1.columns]
jsd_v2   = [jsd_col(tab[c], tab_v2[c]) for c in num_eval if c in tab_v2.columns]

print(f"JSD mediana — v1 (nb05):       {np.median(jsd_v1):.4f}")
print(f"JSD mediana — v2 (este nb):    {np.median(jsd_v2):.4f}")
print(f"Δ JSD mediana:                 {np.median(jsd_v2) - np.median(jsd_v1):+.4f}")

In [ ]:
# Histogramas comparativos para las variables más problemáticas en v1
plot_cols = ["heart_rate_mean", "lactate_mean", "albumin_mean",
             "creatinine_mean", "gcs_eye_min", "troponin_mean"]
plot_cols = [c for c in plot_cols if c in tab.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, col in zip(axes, plot_cols):
    lo, hi = tab[col].quantile(0.01), tab[col].quantile(0.99)
    bins   = np.linspace(lo, hi, 60)
    ax.hist(tab[col].clip(lo, hi), bins=bins, density=True,
            alpha=0.5, color="gray", label="Real")
    if col in tab_v1.columns:
        ax.hist(tab_v1[col].clip(lo, hi), bins=bins, density=True,
                alpha=0.45, color="tomato", label="v1 (nb05)")
    if col in tab_v2.columns:
        ax.hist(tab_v2[col].clip(lo, hi), bins=bins, density=True,
                alpha=0.45, color="steelblue", label="v2 (nb05b)")
    jv1 = jsd_col(tab[col], tab_v1[col]) if col in tab_v1.columns else float("nan")
    jv2 = jsd_col(tab[col], tab_v2[col]) if col in tab_v2.columns else float("nan")
    ax.set_title(f"{col}\nJSD v1={jv1:.3f}  v2={jv2:.3f}", fontsize=9)
    ax.tick_params(labelsize=7)

axes[0].legend(fontsize=8)
fig.suptitle("Distribuciones marginales: Real vs TabDDPM v1 vs v2", fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS / "tabddpm_v2_marginals.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: reports/tabddpm_v2_marginals.png")

## 7. Resumen

In [ ]:
print("=" * 60)
print("  RESUMEN — Notebook 05b (TabDDPM ablation)")
print("=" * 60)
checks = [
    ("Cambio principal",           "Per-column QT + clipping"),
    ("input_dim",                  str(INPUT_DIM)),
    ("Parámetros del modelo",      f"{n_params:,}"),
    ("Épocas entrenadas",          str(N_EPOCHS)),
    ("Loss final",                 f"{losses[-1]:.5f}"),
    ("Tiempo entrenamiento",       f"{total_time:.1f} min"),
    ("JSD mediana v1 (nb05)",      f"{np.median(jsd_v1):.4f}"),
    ("JSD mediana v2 (este nb)",   f"{np.median(jsd_v2):.4f}"),
    ("Muestras generadas",         str(synth_df.shape)),
    ("Mortalidad generada",        f"{synth_df[TARGET].mean()*100:.1f}%"),
]
for name, val in checks:
    print(f"  {name:<35} {val}")
print("=" * 60)